[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Files, Paths and Formats](https://johnfisher-ai.github.io/Python-Visual-Guides/files-paths-and-formats.html)

# JSON on Disk


## What you will be able to do

Write Python data to a JSON file and read it back, and say in advance which parts will come back
changed. You will also be able to read a `JSONDecodeError` well enough to find the character
that broke the file.


## The idea

### The problem

The **CSV** notebook ended on the format's largest limitation: every value is text, and nested
data has nowhere to go. A CSV cannot express a record with a list inside it without inventing a
convention nobody else will follow.

JSON solves both. It has numbers, booleans, null, and arbitrary nesting, and it is what almost
every web service returns.

The cost is a new problem. JSON has **fewer types than Python**, so writing Python data to JSON
is a conversion, and some of it is one-way. A tuple goes in and a list comes out. A dictionary
keyed by integers comes back keyed by strings. A date cannot go in at all.

None of that is a bug, and only some of it announces itself.

### What JSON is

> **JSON** is a text format with six types: object, array, string, number, boolean and null.
> Python's `json` module maps them onto `dict`, `list`, `str`, `int` or `float`, `bool` and
> `None`.
>
> `json.dump` writes to a file and `json.load` reads from one. `json.dumps` and `json.loads`
> do the same with strings, and the `s` stands for string.

The whole format is those six types. Everything else you can express in Python has to become one
of them or be refused.

### What the round trip changes

| You write | You get back | Warned |
|---|---|---|
| `tuple` | `list` | no |
| dictionary with `int` keys | keys become `str` | no |
| `set` | `TypeError` | yes |
| `datetime`, `date` | `TypeError` | yes |
| `float("nan")`, `inf` | written, but not valid JSON | no |

The rows marked "no" are the ones to remember. The refusals are easy: they stop you, you fix it.
The silent conversions carry on and surface much later, usually as a lookup that fails on a key
you are certain exists.

### Where you will meet this

The **APIs and JSON** guide is built on this, because JSON is what web services speak.
Configuration files, exported settings and log pipelines all use it too.

It is also the natural format for saving intermediate results in your own work, and for that
you should know exactly what will come back.

### What this notebook covers

- The six types, and the Python types they map to
- `dump` and `load` against `dumps` and `loads`
- Every round-trip change in the table above, run rather than described
- `indent`, `sort_keys` and `ensure_ascii`
- Writing a date, with `default=`, and why it still comes back as text
- `JSONDecodeError`, and using the position it reports
- JSON Lines, for files too large to hold at once
- Three errors, plus a conversion that raises nothing

### A first look

Nothing to run yet.

```python
import json

original = {"tags": ("a", "b"), "counts": {1: "one"}}

restored = json.loads(json.dumps(original))

print(restored)
# {'tags': ['a', 'b'], 'counts': {'1': 'one'}}
```

The tuple became a list and the integer key became a string. Both happened silently, and
`restored == original` is `False`.


## Setup

Four imports and a folder to work in.

- `json` reads and writes the format, and is the subject of this notebook
- `datetime` supplies a value JSON cannot represent, for the section on `default=`
- `Path` builds paths and reads the files back
- `shutil` removes the scratch folder at the end

**Run this cell before the rest of the notebook.**


In [1]:

import json
import datetime
from pathlib import Path
import shutil

scratch = Path("scratch")
scratch.mkdir(exist_ok=True)

print("working in:", scratch, "->", scratch.exists())


working in: scratch -> True


## Worked examples

### The six types


In [2]:

values = [{"a": 1}, [1, 2], "text", 42, 3.5, True, None]

for value in values:
    as_json = json.dumps(value)
    print(f"{str(value):<12} -> {as_json:<12} -> {json.loads(as_json)!r}")


{'a': 1}     -> {"a": 1}     -> {'a': 1}
[1, 2]       -> [1, 2]       -> [1, 2]
text         -> "text"       -> 'text'
42           -> 42           -> 42
3.5          -> 3.5          -> 3.5
True         -> true         -> True
None         -> null         -> None


Note `True` becoming `true` and `None` becoming `null`. Those are JSON's spellings, and they are
lowercase. Hand-writing a JSON file with `True` in it produces a file nothing can read, which is
the second error in this notebook.

### dumps and dump

`dumps` produces a string. `dump` writes to an open file. The same pair exists for reading.


In [3]:

records = [
    {"name": "Ada", "score": 91},
    {"name": "Grace", "score": 88},
]

print(json.dumps(records))


[{"name": "Ada", "score": 91}, {"name": "Grace", "score": 88}]


In [4]:

target = scratch / "scores.json"

with open(target, "w", encoding="utf-8") as f:
    json.dump(records, f, indent=2)

print(target.read_text(encoding="utf-8"))


[
  {
    "name": "Ada",
    "score": 91
  },
  {
    "name": "Grace",
    "score": 88
  }
]


`indent=2` is worth the space for anything a person will open. Without it the whole file is one
line, which is fine for a machine and unreadable in a diff.


In [5]:

with open(target, encoding="utf-8") as f:
    loaded = json.load(f)

print(loaded)
print("same as what we wrote:", loaded == records)


[{'name': 'Ada', 'score': 91}, {'name': 'Grace', 'score': 88}]
same as what we wrote: True


That one came back identical, because every value in it was already a JSON type. The next
section is about the values that are not.


### A tuple goes in, a list comes out


In [6]:

original = ("north", "south", "east")

restored = json.loads(json.dumps(original))

print("in: ", original, type(original).__name__)
print("out:", restored, type(restored).__name__)
print("equal:", original == restored)


in:  ('north', 'south', 'east') tuple
out: ['north', 'south', 'east'] list
equal: False


JSON has one sequence type, the array, so both `list` and `tuple` are written as arrays and both
come back as lists. Nothing warned, and `original == restored` is `False`.

This matters when the tuple was doing a job a list cannot do. A tuple used as a dictionary key,
as the **Tuples and Unpacking** notebook described, stops working after a round trip.

### Integer keys become strings

This is the one that produces the strangest bug.


In [7]:

by_id = {1: "first", 2: "second"}

restored = json.loads(json.dumps(by_id))

print("in: ", by_id)
print("out:", restored)
print("key types:", type(list(by_id)[0]).__name__, "->", type(list(restored)[0]).__name__)


in:  {1: 'first', 2: 'second'}
out: {'1': 'first', '2': 'second'}
key types: int -> str


JSON object keys are always strings. There is no other option in the format, so `1` was written
as `"1"` and came back as `"1"`.

The failure appears later, in code that has nothing to do with JSON:


In [8]:

print("restored[1] would raise. restored['1'] gives:", restored["1"])
print("1 in restored:", 1 in restored)
print("'1' in restored:", "1" in restored)


restored[1] would raise. restored['1'] gives: first
1 in restored: False
'1' in restored: True


A lookup by integer fails on data that plainly contains that key. If you need integer keys,
convert them back explicitly after loading:


In [9]:

fixed = {int(key): value for key, value in restored.items()}

print(fixed, "| lookup by 1:", fixed[1])


{1: 'first', 2: 'second'} | lookup by 1: first


### What JSON cannot represent

Sets and dates are not JSON types, and there is no single obvious way to convert them, so
`json.dumps` raises `TypeError` instead of choosing one for you.


In [10]:

json.dumps({"regions": {"north", "south"}})


TypeError: Object of type set is not JSON serializable

`Object of type set is not JSON serializable`. This is the good outcome: it stopped, and the
message names the type.

Convert deliberately, and the meaning is yours to choose. A set has no order, so decide whether
that matters before writing it as a list.


In [11]:

regions = {"north", "south", "east"}

print(json.dumps({"regions": sorted(regions)}))


{"regions": ["east", "north", "south"]}


### Dates, and the default= argument

Dates are refused for the same reason, and they are common enough to be worth handling once.


In [12]:

def encode_extra(obj):
    """Convert what json cannot handle. Raise for anything unexpected."""
    if isinstance(obj, (datetime.date, datetime.datetime)):
        return obj.isoformat()
    raise TypeError(f"cannot serialize {type(obj).__name__}")


payload = {"when": datetime.date(2026, 3, 1), "what": "reading"}

print(json.dumps(payload, default=encode_extra))


{"when": "2026-03-01", "what": "reading"}


`default=` is called only for values the module cannot handle itself. Raising for anything the
function does not recognize matters: returning `str(obj)` instead would quietly turn every
unexpected object into text, which is how unreadable files get written.

The date survived the write. It does not survive the read.


In [13]:

restored = json.loads(json.dumps(payload, default=encode_extra))

print(restored)
print("when is a", type(restored["when"]).__name__)


{'when': '2026-03-01', 'what': 'reading'}
when is a str


JSON has no date type, so `"2026-03-01"` is a string and comes back as one. Converting on the
way back in is a separate step, and it is yours to write.


In [14]:

when = datetime.date.fromisoformat(restored["when"])

print(when, type(when).__name__, "| year:", when.year)


2026-03-01 date | year: 2026


### NaN and Infinity are written and are not valid JSON

This one is worth knowing because the file looks fine until something else reads it.


In [15]:

measurements = {"value": float("nan"), "limit": float("inf")}

text = json.dumps(measurements)

print("Python wrote:", text)
print("and reads it back:", json.loads(text))


Python wrote: {"value": NaN, "limit": Infinity}
and reads it back: {'value': nan, 'limit': inf}


`NaN` and `Infinity` are not in the JSON specification. Python writes them anyway, and reads its
own output, so a round trip inside Python succeeds.

Hand that file to a browser, a database loader or another language and it will be rejected.
Pass `allow_nan=False` to find out at the point of writing instead.


In [16]:

json.dumps(measurements, allow_nan=False)


ValueError: Out of range float values are not JSON compliant: nan

`Out of range float values are not JSON compliant: nan`. Decide what those values should be,
usually `None`, and convert them before writing.


### Formatting


In [17]:

data = {"b": 2, "a": 1, "note": "caf\u00e9"}

print("default:           ", json.dumps(data))
print("sort_keys=True:    ", json.dumps(data, sort_keys=True))
print("ensure_ascii=False:", json.dumps(data, ensure_ascii=False))


default:            {"b": 2, "a": 1, "note": "caf\u00e9"}
sort_keys=True:     {"a": 1, "b": 2, "note": "caf\u00e9"}
ensure_ascii=False: {"b": 2, "a": 1, "note": "café"}


By default the module escapes every non-ASCII character, so `café` is written as `caf\u00e9`.
That is valid JSON and readable by anything, and it is unreadable to a person.

`ensure_ascii=False` writes the character itself, which is what you want for a file anyone will
open. Combine it with `encoding="utf-8"` on the file, since the escaping was the thing making
the ASCII-only version safe.

`sort_keys=True` is worth using for anything stored in version control: without it the key order
follows insertion, and an unrelated change reorders the whole file in the diff.


### JSON Lines, for files too big to load

`json.load` reads the whole file into memory. For a large log that is a problem, and the
convention is one JSON object per line.


In [18]:

events = scratch / "events.jsonl"

with open(events, "w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record) + "\n")

print(repr(events.read_text(encoding="utf-8")))


'{"name": "Ada", "score": 91}\n{"name": "Grace", "score": 88}\n'


In [19]:

with open(events, encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        print(record["name"], record["score"])


Ada 91
Grace 88


One line at a time, one record in memory at a time, and the file can be any size. The extension
is usually `.jsonl` or `.ndjson`.

Note that a JSON Lines file is not valid JSON. Each line is; the file as a whole is not, and
`json.load` on it fails.


## Your turn

Six tasks. Write your answer in the cell under each and run it.

Try each one before you look at an answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/files-paths-and-formats/05-json-on-disk-solutions.ipynb).

**1.** Write a list of two dictionaries to `scratch/people.json` with `indent=2`, then print the
file text.


In [20]:
# your code here


**2.** Load it back and confirm it equals what you wrote.


In [21]:
# your code here


**3.** Round-trip `{"point": (3, 7), "ids": {10: "a"}}` and print the result. Say in a comment
what changed.


In [22]:
# your code here


**4.** Try to write `{"seen": {1, 2, 3}}` and print the exception type and message instead of
letting it stop the cell. Then write it successfully.


In [23]:
# your code here


**5.** Write `{"today": datetime.date.today(), "note": "x"}` using a `default=` function, then
load it and convert the date back to a `date`.


In [24]:
# your code here


**6.** Write three records as JSON Lines, then read the file back one line at a time and print
each record's name.


In [25]:
# your code here


## Common errors

Each cell below is run on purpose so you can see the real message.

### JSONDecodeError: the file is not valid JSON

The message carries a position, and the position is the useful part.


In [26]:

json.loads('{"name": "Ada", "score": 91,}')


JSONDecodeError: Illegal trailing comma before end of object: line 1 column 28 (char 27)

`Illegal trailing comma before end of object: line 1 column 28`. A trailing comma is legal in
Python and not in JSON, and it is the most common way a hand-edited file breaks.

Three more, caught rather than raised, so you can compare the messages:


In [27]:

for text in ["{'name': 'Ada'}", '{"name": "Ada"', "not json at all", '{"ok": True}']:
    try:
        json.loads(text)
    except json.JSONDecodeError as e:
        print(f"{text!r:<22} line {e.lineno} col {e.colno}: {e.msg}")


"{'name': 'Ada'}"      line 1 col 2: Expecting property name enclosed in double quotes
'{"name": "Ada"'       line 1 col 15: Expecting ',' delimiter
'not json at all'      line 1 col 1: Expecting value
'{"ok": True}'         line 1 col 8: Expecting value


Reading them in order: JSON requires double quotes, so single quotes fail on the first
character of the key. An unclosed brace fails at the end. Text that is not JSON at all fails
immediately. And `True` fails because JSON spells it `true`.

That last one catches people who write a JSON file by printing a Python dictionary. `json.dumps`
exists so you never have to.


### TypeError: that object has no JSON equivalent


In [28]:

json.dumps({"when": datetime.datetime.now()})


TypeError: Object of type datetime is not JSON serializable

`Object of type datetime is not JSON serializable`. Raising is the right behavior here: there are several reasonable ways to write a datetime and it does not
know which you want.

Pass `default=` as shown above, and decide deliberately.


### AttributeError: load against loads

The names are one letter apart and do different things.


In [29]:

text = '{"a": 1}'

json.load(text)


AttributeError: 'str' object has no attribute 'read'

`'str' object has no attribute 'read'`. `json.load` expects an open file and was handed a
string, so it tried to call `.read()` on it.

`loads` takes the string. The rule: **`s` means string**, for both `dumps` and `loads`.


In [30]:

print(json.loads(text))


{'a': 1}


### The quiet one: the round trip that changed your data


In [31]:

config = {
    "thresholds": {1: 0.5, 2: 0.8},
    "regions": ("north", "south"),
}

saved = scratch / "config.json"
saved.write_text(json.dumps(config), encoding="utf-8")

restored = json.loads(saved.read_text(encoding="utf-8"))

print("wrote:   ", config)
print("restored:", restored)
print("equal:", config == restored)


wrote:    {'thresholds': {1: 0.5, 2: 0.8}, 'regions': ('north', 'south')}
restored: {'thresholds': {'1': 0.5, '2': 0.8}, 'regions': ['north', 'south']}
equal: False


Nothing raised at any point. The file is valid JSON, it loaded cleanly, and the data is not what
was written: the keys are strings and the tuple is a list.

A program that saves state and reloads it will work perfectly on the first run and behave
differently on the second, which is a hard bug to see because the code is identical both times.

Two defenses. Keep what you write to JSON inside JSON's types in the first place, using strings
for keys and lists for sequences. Or write the conversion back explicitly, as a function that
runs on load.


In [32]:

def restore(raw):
    return {
        "thresholds": {int(k): v for k, v in raw["thresholds"].items()},
        "regions": tuple(raw["regions"]),
    }

print(restore(restored) == config)


True


### Cleaning up


In [33]:

shutil.rmtree(scratch)

print("scratch still there:", scratch.exists())


scratch still there: False


## Recap

- JSON has six types. Anything else is converted or refused.
- `dump`/`load` use files; `dumps`/`loads` use strings, and the `s` is the difference.
- A **tuple becomes a list** and **integer keys become strings**, both without warning.
- Sets and dates raise `TypeError`, which is the helpful outcome.
- `default=` handles what the module cannot, and should raise for anything it does not expect.
- A date written as text comes back as text. Converting back is a separate step.
- `NaN` and `Infinity` are written by Python and rejected by everything else; use
  `allow_nan=False` to find out early.
- `indent` for readability, `sort_keys` for diffs, `ensure_ascii=False` for people.
- JSON Lines holds one object per line, for files too large to load at once.


## What is next

The **Excel Files** notebook, which reads the format most data actually arrives in when a person
rather than a program produced it, and explains why a date from a spreadsheet often turns up as
a five-digit number.


---

&#8592; **Previous:** [CSV](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/files-paths-and-formats/04-csv.ipynb)  &nbsp;·&nbsp;  [Files, Paths and Formats Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/files-paths-and-formats.html)
